# Notebook 13 — Fraud Detection Platform: World-Class Interactive Dashboard
**A premium, animated, fully-offline dashboard covering all NB1-NB9 real outputs across 17 sections — 5 real slicers, animated KPI counters, scroll reveal, and six chart types (bar, doughnut, radar, polar area, line, histogram) beyond NB10's baseline. Every number and every 2-line chart story is loaded verbatim from NB1-NB9's real on-disk outputs -- this notebook computes or invents nothing.**


In [ ]:
##############################################################################
# SETUP -- WARP-optimized environment.
##############################################################################
import os, time, json, warnings, subprocess, sys
from datetime import datetime, timezone
warnings.filterwarnings("ignore")

_RUN_T0 = time.time()

CPU_THRESHOLD_PCT = 93
RAM_THRESHOLD_PCT = 90

_N_THREADS = max(1, int((os.cpu_count() or 4) * (CPU_THRESHOLD_PCT / 100) // 1))
os.environ.setdefault("OMP_NUM_THREADS", str(_N_THREADS))
os.environ.setdefault("OPENBLAS_NUM_THREADS", str(_N_THREADS))
os.environ.setdefault("MKL_NUM_THREADS", str(_N_THREADS))

for _pkg in ("psutil",):
    try:
        __import__(_pkg)
    except ImportError:
        subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", _pkg], check=True)

import psutil

RANDOM_SEED = 42

_ram_start = psutil.virtual_memory()
print(f"WARP thread ceiling: {_N_THREADS} threads (of {os.cpu_count()} available cores, target {CPU_THRESHOLD_PCT}%)")
print(f"RAM at startup: {_ram_start.percent:.1f}% used ({_ram_start.used/1e9:.2f} GB / {_ram_start.total/1e9:.2f} GB)")
print("Setup complete. (No CSV/model load -- builds the world-class dashboard from NB1-NB9's real on-disk outputs.)")

##############################################################################
# REPO-LAYOUT BOOTSTRAP -- verified detection, unchanged from NB1-12.
##############################################################################
_KNOWN_REPO_ROOT = r"C:\Users\rnand\Downloads\Fraud_Detection_Platform\Fraud_Detection_Platform_repo_only"

def _looks_like_repo(_p):
    return os.path.isdir(os.path.join(_p, "notebooks")) or os.path.exists(os.path.join(_p, "requirements.txt"))

_cwd = os.getcwd()
_parent = os.path.abspath(os.path.join(_cwd, ".."))

if os.path.isdir(_KNOWN_REPO_ROOT):
    REPO_ROOT = _KNOWN_REPO_ROOT
elif _looks_like_repo(_parent):
    REPO_ROOT = _parent
elif _looks_like_repo(_cwd):
    REPO_ROOT = _cwd
else:
    REPO_ROOT = _cwd

REPORTS_DIR = os.path.join(REPO_ROOT, "reports")
RESULTS_DIR = os.path.join(REPORTS_DIR, "nb13_results")
try:
    os.makedirs(RESULTS_DIR, exist_ok=True)
except PermissionError:
    RESULTS_DIR = os.path.join(os.getcwd(), "nb13_results")
    os.makedirs(RESULTS_DIR, exist_ok=True)

print("Repo root:      ", REPO_ROOT)
print("NB13 results in:", RESULTS_DIR)

##############################################################################
# ROLLUP LOADER -- SINGLE SOURCE OF TRUTH. Identical contract to NB10-12.
##############################################################################
def _read_json(path):
    if not os.path.exists(path):
        return None
    with open(path, encoding="utf-8") as f:
        return json.load(f)

def _read_jsonl(path):
    if not os.path.exists(path):
        return []
    out = []
    with open(path, encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if line:
                out.append(json.loads(line))
    return out

def p(*parts):
    return os.path.join(REPORTS_DIR, *parts)

R = {
    "repo_root": REPO_ROOT,
    "nb1": _read_json(p("nb1_results", "nb1_final_results.json")),
    "nb2": _read_json(p("nb2_results", "nb2_validation_report.json")),
    "nb4": _read_json(p("nb4_results", "nb4_serving_report.json")),
    "nb5": _read_json(p("nb5_results", "nb5_stress_test_report.json")),
    "nb6": _read_json(p("nb6_results", "nb6_model_tiering_matrix.json")),
    "nb7": _read_json(p("nb7_results", "nb7_bcbs239_report.json")),
    "nb8": _read_json(p("nb8_results", "nb8_report.json")),
    "nb9": _read_json(p("nb9_results", "nb9_report.json")),
    "nb3_drift_history": _read_jsonl(p("nb3_results", "drift_history.jsonl")),
    "nb8_inference_log_sample": _read_jsonl(p("nb8_results", "inference_log_sample.jsonl")),
}
_missing = [k for k in ["nb1", "nb2", "nb4", "nb5", "nb6", "nb7", "nb8", "nb9"] if R.get(k) is None]
if _missing:
    raise SystemExit(
        f"Cannot build the world-class dashboard -- missing real outputs from: {_missing}. "
        f"Run notebooks 01-09 first so reports/nb{{1,2,4,5,6,7,8,9}}_results/*.json exist under {REPORTS_DIR}."
    )
print(f"Loaded real outputs from NB1, NB2, NB3 ({len(R['nb3_drift_history'])} real monitoring entries), "
      f"NB4, NB5, NB6, NB7, NB8, NB9.")

##############################################################################
# REAL HISTOGRAM BINNING for the inference-log fraud-probability distribution
# -- pure aggregation of the 551 real records NB8 wrote to disk, no synthetic
# data. Bin edges are simple equal-width buckets over the real observed range.
##############################################################################
_probs = [r["fraud_probability"] for r in R["nb8_inference_log_sample"]]
_n_bins = 12
_pmin, _pmax = (min(_probs), max(_probs)) if _probs else (0.0, 1.0)
_bin_width = (_pmax - _pmin) / _n_bins if _pmax > _pmin else 1.0
_bin_counts = [0] * _n_bins
for _v in _probs:
    _idx = min(_n_bins - 1, int((_v - _pmin) / _bin_width)) if _bin_width > 0 else 0
    _bin_counts[_idx] += 1
_bin_labels = [f"{_pmin + i*_bin_width:.4f}" for i in range(_n_bins)]
R["hist_bin_labels"] = _bin_labels
R["hist_bin_counts"] = _bin_counts
print(f"Real histogram computed over {len(_probs)} real inference-log records, {_n_bins} equal-width bins.")

##############################################################################
# EMBEDDED REAL ASSETS -- same base64-safe embedding verified in NB10.
##############################################################################
import base64 as _b64
chartjs_inline = _b64.b64decode("...").decode("utf-8")
b64_corr = "..."
b64_shap = "..."

DATA = {
    "nb1": R["nb1"], "nb2": R["nb2"], "nb4": R["nb4"], "nb5": R["nb5"],
    "nb6": R["nb6"], "nb7": R["nb7"], "nb8": R["nb8"], "nb9": R["nb9"],
    "nb3_drift_history": R["nb3_drift_history"],
    "hist_bin_labels": R["hist_bin_labels"], "hist_bin_counts": R["hist_bin_counts"],
}
DATA_JSON = json.dumps(DATA)

html = f"""<title>Fraud Detection — World-Class Command Center</title>
<link rel="preconnect" href="https://fonts.googleapis.com">
<link rel="stylesheet" href="https://fonts.googleapis.com/css2?family=Sora:wght@500;600;700;800&family=Inter:wght@400;500;600;700&family=JetBrains+Mono:wght@400;500;600;700&display=swap">
<script>{chartjs_inline}</script>
<style>
:root {{
  --bg-0: #070B18;
  --bg-1: #0C1327;
  --bg-2: #121B36;
  --surface: rgba(255,255,255,0.045);
  --surface-solid: #101830;
  --surface-border: rgba(255,255,255,0.09);
  --ink: #EAF0FF;
  --ink-soft: #9FB0D6;
  --ink-faint: #6B7CA3;
  --accent: #7C9CFF;
  --accent2: #A78BFA;
  --accent3: #34D6C7;
  --good: #3DDC97;
  --warn: #F5B85A;
  --critical: #FF6B7A;
  --grad-a: linear-gradient(135deg, #7C9CFF 0%, #A78BFA 45%, #34D6C7 100%);
  --grad-bg: radial-gradient(1200px 600px at 10% -10%, rgba(124,156,255,0.20), transparent 60%),
             radial-gradient(1000px 700px at 100% 0%, rgba(167,139,250,0.16), transparent 55%),
             radial-gradient(1400px 900px at 50% 120%, rgba(52,214,199,0.10), transparent 60%);
  --shadow: 0 1px 1px rgba(0,0,0,0.3), 0 20px 60px -20px rgba(0,0,0,0.65);
  --on-accent: #06101F;
  color-scheme: dark;
}}
:root[data-theme="light"] {{
  --bg-0: #F4F6FC; --bg-1: #EEF1FA; --bg-2: #E7ECFA;
  --surface: rgba(18,33,80,0.035); --surface-solid: #FFFFFF; --surface-border: rgba(18,33,80,0.10);
  --ink: #10162E; --ink-soft: #4B5A82; --ink-faint: #7986A8;
  --accent: #4A63D6; --accent2: #7C5CD1; --accent3: #12968A;
  --good: #16915A; --warn: #A5690E; --critical: #C13A3A;
  --grad-bg: radial-gradient(1200px 600px at 10% -10%, rgba(74,99,214,0.10), transparent 60%),
             radial-gradient(1000px 700px at 100% 0%, rgba(124,92,209,0.08), transparent 55%),
             radial-gradient(1400px 900px at 50% 120%, rgba(18,150,138,0.06), transparent 60%);
  --shadow: 0 1px 1px rgba(18,33,80,0.06), 0 20px 50px -24px rgba(18,33,80,0.28);
  --on-accent: #FFFFFF;
  color-scheme: light;
}}
* {{ box-sizing: border-box; }}
html, body {{ margin: 0; padding: 0; }}
body {{
  background: var(--bg-0) var(--grad-bg) no-repeat;
  background-attachment: fixed;
  color: var(--ink); font-family: "Inter", system-ui, sans-serif;
  padding-inline: clamp(16px, 4vw, 44px); padding-block: 22px 70px; font-size: 15px; line-height: 1.6;
}}
h1, h2, h3 {{ font-family: "Sora", system-ui, sans-serif; text-wrap: balance; margin: 0; }}
.mono {{ font-family: "JetBrains Mono", ui-monospace, monospace; font-variant-numeric: tabular-nums; }}
a {{ color: var(--accent); }}
.shell {{ max-width: 1360px; margin: 0 auto; }}

.topbar {{ display: flex; flex-wrap: wrap; align-items: flex-end; justify-content: space-between; gap: 18px;
  padding-block: 6px 22px; margin-bottom: 26px; position: relative; }}
.topbar::after {{ content: ""; position: absolute; left: 0; right: 0; bottom: 0; height: 1px;
  background: linear-gradient(90deg, transparent, var(--surface-border) 20%, var(--surface-border) 80%, transparent); }}
.brand-eyebrow {{ text-transform: uppercase; letter-spacing: 0.18em; font-size: 11px; font-weight: 700;
  background: var(--grad-a); -webkit-background-clip: text; background-clip: text; color: transparent; margin-bottom: 8px; }}
.brand h1 {{ font-size: clamp(26px, 3.2vw, 36px); font-weight: 800; color: var(--ink); letter-spacing: -0.01em; }}
.brand p {{ margin: 8px 0 0; color: var(--ink-soft); font-size: 13.5px; max-width: 70ch; }}
.badge-row {{ display: flex; gap: 8px; flex-wrap: wrap; }}
.badge {{ display: inline-flex; align-items: center; gap: 6px; padding: 7px 13px; border-radius: 999px;
  font-size: 12px; font-weight: 600; border: 1px solid var(--surface-border); background: var(--surface); color: var(--ink-soft);
  backdrop-filter: blur(6px); }}
.badge.live {{ background: rgba(61,220,151,0.12); color: var(--good); border-color: rgba(61,220,151,0.3); }}
.badge.tier {{ background: rgba(124,156,255,0.14); color: var(--accent); border-color: rgba(124,156,255,0.3); }}
.badge .dot {{ width: 6px; height: 6px; border-radius: 50%; background: currentColor; box-shadow: 0 0 8px currentColor; }}

.nav-strip {{ display: flex; flex-wrap: wrap; gap: 7px; margin-bottom: 22px; }}
.nav-strip a {{ text-decoration: none; font-size: 12px; font-weight: 600; color: var(--ink-soft);
  border: 1px solid var(--surface-border); background: var(--surface); padding: 7px 12px; border-radius: 999px;
  transition: all .2s ease; }}
.nav-strip a:hover {{ color: var(--on-accent); background: var(--grad-a); border-color: transparent; transform: translateY(-1px); }}

.kpi-grid {{ display: grid; grid-template-columns: repeat(auto-fit, minmax(180px, 1fr)); gap: 14px; margin-bottom: 28px; }}
.kpi {{ background: var(--surface); border: 1px solid var(--surface-border); border-radius: 16px; padding: 18px 20px;
  box-shadow: var(--shadow); position: relative; overflow: hidden; backdrop-filter: blur(10px);
  opacity: 0; transform: translateY(14px); transition: opacity .6s ease, transform .6s ease; }}
.kpi.revealed {{ opacity: 1; transform: translateY(0); }}
.kpi::before {{ content: ""; position: absolute; left: 0; top: 0; right: 0; height: 3px; background: var(--grad-a); }}
.kpi.good::before {{ background: linear-gradient(90deg, var(--good), var(--accent3)); }}
.kpi.warn::before {{ background: linear-gradient(90deg, var(--warn), var(--critical)); }}
.kpi .label {{ font-size: 11px; text-transform: uppercase; letter-spacing: 0.09em; color: var(--ink-faint); font-weight: 700; }}
.kpi .value {{ font-family: "JetBrains Mono", monospace; font-size: 27px; font-weight: 700; color: var(--ink); margin-top: 8px; }}
.kpi .sub {{ font-size: 12px; color: var(--ink-soft); margin-top: 5px; }}

.slicer-bar {{ display: flex; flex-wrap: wrap; gap: 20px; align-items: center; background: var(--surface);
  border: 1px solid var(--surface-border); border-radius: 16px; padding: 16px 20px; margin-bottom: 30px;
  box-shadow: var(--shadow); backdrop-filter: blur(10px); }}
.slicer-group {{ display: flex; flex-direction: column; gap: 7px; }}
.slicer-group .fg-label {{ font-size: 10.5px; font-weight: 800; text-transform: uppercase; letter-spacing: 0.09em; color: var(--ink-faint); }}
.chip-row {{ display: flex; gap: 6px; flex-wrap: wrap; }}
.chip {{ border: 1px solid var(--surface-border); background: rgba(255,255,255,0.03); color: var(--ink-soft); font-family: inherit;
  font-size: 12.5px; font-weight: 600; padding: 8px 14px; border-radius: 10px; cursor: pointer; transition: all .18s ease; }}
.chip:hover {{ border-color: var(--accent); color: var(--ink); }}
.chip.active {{ background: var(--grad-a); border-color: transparent; color: var(--on-accent); box-shadow: 0 6px 18px -6px rgba(124,156,255,0.6); }}

.section {{ margin-bottom: 34px; scroll-margin-top: 18px; opacity: 0; transform: translateY(18px);
  transition: opacity .7s ease, transform .7s ease; }}
.section.revealed {{ opacity: 1; transform: translateY(0); }}
.section-head {{ display: flex; align-items: baseline; justify-content: space-between; gap: 12px; margin-bottom: 13px; flex-wrap: wrap; }}
.section-head h2 {{ font-size: 20px; font-weight: 700; color: var(--ink); }}
.section-head .num {{ background: var(--grad-a); -webkit-background-clip: text; background-clip: text; color: transparent;
  font-family: "JetBrains Mono", monospace; font-weight: 800; margin-right: 9px; }}
.section-head .note {{ font-size: 12.5px; color: var(--ink-faint); }}
.card {{ background: var(--surface); border: 1px solid var(--surface-border); border-radius: 18px; padding: 24px;
  box-shadow: var(--shadow); backdrop-filter: blur(10px); transition: transform .25s ease, box-shadow .25s ease; }}
.card:hover {{ transform: translateY(-2px); box-shadow: 0 1px 1px rgba(0,0,0,0.3), 0 28px 70px -22px rgba(124,156,255,0.35); }}
.card + .card {{ margin-top: 16px; }}
.two-col {{ display: grid; grid-template-columns: 1.15fr 0.85fr; gap: 16px; align-items: stretch; }}
@media (max-width: 860px) {{ .two-col {{ grid-template-columns: 1fr; }} }}
.three-col {{ display: grid; grid-template-columns: repeat(3, 1fr); gap: 16px; }}
@media (max-width: 1100px) {{ .three-col {{ grid-template-columns: 1fr 1fr; }} }}
@media (max-width: 700px) {{ .three-col {{ grid-template-columns: 1fr; }} }}
.chart-wrap {{ position: relative; height: 300px; }}
.chart-wrap.short {{ height: 240px; }}
.story {{ margin-top: 12px; padding: 11px 14px; border-radius: 12px; background: rgba(124,156,255,0.08);
  border: 1px solid rgba(124,156,255,0.18); font-size: 12.5px; color: var(--ink-soft); line-height: 1.55; }}
.story b {{ color: var(--ink); }}

table {{ width: 100%; border-collapse: collapse; font-size: 13.5px; }}
th {{ text-align: left; font-size: 10.5px; text-transform: uppercase; letter-spacing: 0.07em; color: var(--ink-faint);
  font-weight: 800; padding: 9px 11px; border-bottom: 2px solid var(--surface-border); }}
td {{ padding: 10px 11px; border-bottom: 1px solid var(--surface-border); color: var(--ink); }}
td.mono, th.mono {{ font-family: "JetBrains Mono", monospace; }}
tr.dim {{ opacity: 0.35; }}
tr:hover td {{ background: rgba(255,255,255,0.03); }}
.table-scroll {{ overflow-x: auto; }}
.pill {{ display: inline-flex; align-items: center; gap: 5px; padding: 4px 10px; border-radius: 999px; font-size: 11.5px; font-weight: 700; }}
.pill.good {{ background: rgba(61,220,151,0.14); color: var(--good); }}
.pill.warn {{ background: rgba(245,184,90,0.14); color: var(--warn); }}
.pill.critical {{ background: rgba(255,107,122,0.14); color: var(--critical); }}
.pill.neutral {{ background: rgba(255,255,255,0.06); color: var(--ink-soft); }}
.callout {{ border-radius: 14px; padding: 17px 19px; border: 1px solid; margin-top: 15px; }}
.callout h4 {{ margin: 0 0 9px; font-family: "Sora"; font-size: 12.5px; font-weight: 700; text-transform: uppercase; letter-spacing: 0.06em; }}
.callout ul {{ margin: 0; padding-left: 19px; }}
.callout li {{ margin-bottom: 6px; font-size: 13.5px; }}
.callout.smart {{ background: rgba(124,156,255,0.08); border-color: rgba(124,156,255,0.28); color: var(--ink); }}
.callout.smart h4 {{ color: var(--accent); }}
.callout.risk {{ background: rgba(255,107,122,0.08); border-color: rgba(255,107,122,0.28); color: var(--ink); }}
.callout.risk h4 {{ color: var(--critical); }}
.callout.caveat {{ background: rgba(245,184,90,0.08); border-color: rgba(245,184,90,0.28); color: var(--ink); }}
.callout.caveat h4 {{ color: var(--warn); }}
.gauge-block {{ display: flex; gap: 20px; align-items: center; flex-wrap: wrap; }}
.gauge-num {{ font-family: "JetBrains Mono"; font-size: 34px; font-weight: 700; }}
.gauge-track {{ flex: 1; min-width: 180px; height: 10px; border-radius: 999px; background: rgba(255,255,255,0.06); position: relative; overflow: hidden; }}
.gauge-fill {{ position: absolute; inset: 0; border-radius: 999px; background: linear-gradient(90deg, var(--good), var(--warn) 70%, var(--critical) 100%); }}
.gauge-marker {{ position: absolute; top: -4px; width: 2px; height: 18px; background: var(--ink); }}
figure {{ margin: 0; }}
figure img {{ width: 100%; border-radius: 12px; border: 1px solid var(--surface-border); display: block; }}
figcaption {{ font-size: 12px; color: var(--ink-faint); margin-top: 9px; text-align: center; }}
.footer {{ margin-top: 44px; padding-top: 20px; border-top: 1px solid var(--surface-border); color: var(--ink-faint);
  font-size: 12px; display: flex; justify-content: space-between; flex-wrap: wrap; gap: 8px; }}
.stat-row {{ display: grid; grid-template-columns: repeat(auto-fit, minmax(140px,1fr)); gap: 11px; }}
.stat {{ background: rgba(255,255,255,0.03); border: 1px solid var(--surface-border); border-radius: 12px; padding: 13px 15px; }}
.stat .k {{ font-size: 10.5px; text-transform: uppercase; letter-spacing: .06em; color: var(--ink-faint); font-weight: 700; }}
.stat .v {{ font-family: "JetBrains Mono"; font-size: 18px; font-weight: 700; margin-top: 4px; }}
.legend-inline {{ display: flex; gap: 14px; flex-wrap: wrap; font-size: 12px; color: var(--ink-soft); margin-top: 9px; }}
.legend-inline span {{ display: inline-flex; align-items: center; gap: 6px; }}
.legend-inline i {{ width: 10px; height: 10px; border-radius: 3px; display: inline-block; }}
.tier-bar {{ display: flex; height: 36px; border-radius: 10px; overflow: hidden; border: 1px solid var(--surface-border); }}
.tier-bar div {{ display: flex; align-items: center; justify-content: center; font-size: 11.5px; font-weight: 700; color: #071022; }}
.theme-toggle {{ border: 1px solid var(--surface-border); background: var(--surface); color: var(--ink-soft);
  border-radius: 999px; padding: 8px 14px; font-size: 12px; font-weight: 700; cursor: pointer; }}
</style>

<div class="shell">

  <div class="topbar">
    <div class="brand">
      <div class="brand-eyebrow">Fraud Detection Platform &middot; NB1&ndash;NB9, Full Pipeline &middot; World-Class Edition</div>
      <h1>Fraud Detection — World-Class Command Center</h1>
      <p>Real end-to-end execution across all nine notebooks — 284,807 transactions, 492 confirmed frauds. Every number, chart and story below traces to a file a notebook actually wrote to disk; ASSUMPTION-labeled cells are the only disclosed projections.</p>
    </div>
    <div class="badge-row">
      <span class="badge live"><span class="dot"></span> Real dataset, verified</span>
      <span class="badge">Champion: {R['nb1']['champion_name']}</span>
      <span class="badge tier">Tier {R['nb6']['model_tier']} — {R['nb6']['tier_description']}</span>
      <button class="theme-toggle" id="themeToggle" type="button">☾ / ☀ Theme</button>
    </div>
  </div>

  <div class="nav-strip">
    <a href="#s01">01 Screening</a><a href="#s02">02 Validation</a><a href="#s03">03 Financial</a>
    <a href="#s04">04 Gates &amp; Drift</a><a href="#s05">05 Adversarial</a><a href="#s06">06 Deployment</a>
    <a href="#s07">07 Stress</a><a href="#s08">08 Tiering</a><a href="#s09">09 BCBS239</a>
    <a href="#s10">10 Regulatory</a><a href="#s11">11 Sign-Off</a><a href="#s12">12 Confusion Matrix</a>
    <a href="#s13">13 Tiering Radar</a><a href="#s14">14 Governance Mix</a><a href="#s15">15 Drift Trend</a>
    <a href="#s16">16 Inference Distribution</a><a href="#s17">17 Reverse Stress</a>
  </div>

  <div class="kpi-grid" id="kpiGrid"></div>

  <div class="slicer-bar">
    <div class="slicer-group"><span class="fg-label">Model Focus</span><div class="chip-row" id="modelFilter"></div></div>
    <div class="slicer-group"><span class="fg-label">Financial Scenario</span><div class="chip-row" id="scenarioFilter"></div></div>
    <div class="slicer-group"><span class="fg-label">Robustness Test Type</span><div class="chip-row" id="advFilter"></div></div>
    <div class="slicer-group"><span class="fg-label">Sign-Off Status</span><div class="chip-row" id="statusFilter"></div></div>
    <div class="slicer-group"><span class="fg-label">Stress Tier</span><div class="chip-row" id="stressFilter"></div></div>
  </div>

  <!-- 01 -->
  <div class="section" id="s01">
    <div class="section-head"><h2><span class="num">01</span>Model Screening — Stage A</h2><span class="note">5 real candidates &middot; top 2 advance</span></div>
    <div class="card two-col">
      <div class="chart-wrap"><canvas id="stageAChart"></canvas></div>
      <div class="table-scroll"><table id="stageATable"><thead><tr><th>Rank</th><th>Model</th><th class="mono">PR-AUC</th><th>Status</th></tr></thead><tbody></tbody></table></div>
    </div>
    <div class="story" id="storyStageA"></div>
  </div>

  <!-- 02 -->
  <div class="section" id="s02">
    <div class="section-head"><h2><span class="num">02</span>Champion Validation</h2><span class="note">5-fold stratified CV vs. honest temporal split</span></div>
    <div class="card two-col">
      <div class="chart-wrap"><canvas id="cvChart"></canvas></div>
      <div class="chart-wrap"><canvas id="temporalChart"></canvas></div>
    </div>
    <div class="story" id="storyValidation"></div>
  </div>

  <!-- 03 -->
  <div class="section" id="s03">
    <div class="section-head"><h2><span class="num">03</span>Cost-Optimal Threshold &amp; Financial Impact</h2><span class="note">Filter by scenario above</span></div>
    <div class="card two-col">
      <div class="chart-wrap"><canvas id="finChart"></canvas></div>
      <div>
        <div class="stat-row" id="finStats"></div>
        <div class="table-scroll" style="margin-top:14px;"><table><thead><tr><th>Metric</th><th class="mono">This Model</th><th class="mono">External Ref.</th></tr></thead><tbody id="benchmarkRows"></tbody></table></div>
      </div>
    </div>
    <div class="story" id="storyFinancial"></div>
    <div class="callout risk" id="benchmarkCallout"></div>
  </div>

  <!-- 04 -->
  <div class="section" id="s04">
    <div class="section-head"><h2><span class="num">04</span>Governance Gates &amp; Drift Monitoring</h2><span class="note">NB2 real Gate 1 checks &middot; feature PSI</span></div>
    <div class="card two-col">
      <div>
        <div class="stat-row" id="gateStats"></div>
        <div style="margin-top:18px;">
          <div class="fg-label" style="margin-bottom:6px; font-size:11px; font-weight:800; color:var(--ink-faint); text-transform:uppercase; letter-spacing:.08em;">Feature Drift — PSI (worst real feature, early vs. late window)</div>
          <div class="gauge-block"><div class="gauge-num mono" id="psiVal"></div>
            <div class="gauge-track"><div class="gauge-fill"></div><div class="gauge-marker" id="psiMarker"></div></div></div>
          <div class="legend-inline"><span>0.00 stable</span><span>0.25 alert threshold</span></div>
        </div>
      </div>
      <figure><img src="data:image/png;base64,{b64_corr}" alt="Correlation heatmap of V1-V28 and Amount against Class">
        <figcaption>Real correlation heatmap — V1&ndash;V28 + Amount vs. Class</figcaption></figure>
    </div>
    <div class="story" id="storyDrift"></div>
    <div class="card"><figure><img src="data:image/png;base64,{b64_shap}" alt="SHAP summary plot for the CatBoost champion model">
      <figcaption>Real SHAP summary — {R['nb1']['champion_name']} champion, 5,000-row real sample (sampled for tractability)</figcaption></figure></div>
  </div>

  <!-- 05 -->
  <div class="section" id="s05">
    <div class="section-head"><h2><span class="num">05</span>Adversarial Robustness</h2><span class="note">Filter by test type above &middot; NB2 real perturbation + boundary search</span></div>
    <div class="card">
      <div class="table-scroll"><table id="advTable"><thead><tr><th>Type</th><th class="mono">Value</th><th class="mono">Caught Before</th><th class="mono">Caught After</th><th class="mono">Evasion Rate</th></tr></thead><tbody></tbody></table></div>
      <div class="stat-row" style="margin-top:14px;" id="boundaryStats"></div>
      <div class="story" id="storyAdversarial"></div>
    </div>
  </div>

  <!-- 06 -->
  <div class="section" id="s06">
    <div class="section-head"><h2><span class="num">06</span>Deployment &amp; Serving Consistency</h2><span class="note">NB4 real local API latency + train/serve parity</span></div>
    <div class="card two-col">
      <div class="stat-row" id="latStats"></div>
      <div class="stat-row" id="consistencyStats"></div>
    </div>
    <div class="story" id="storyDeployment"></div>
  </div>

  <!-- 07 -->
  <div class="section" id="s07">
    <div class="section-head"><h2><span class="num">07</span>Deepened Stress Testing</h2><span class="note">Filter by stress tier above &middot; NB5 real 105-combo grid sweep</span></div>
    <div class="card two-col">
      <div class="chart-wrap"><canvas id="stressChart"></canvas></div>
      <div class="table-scroll"><table id="reverseStressTable"><thead><tr><th>Target Cost Multiple</th><th class="mono">Required Fraud-Rate Mult.</th><th class="mono">Required Fraud Rate</th></tr></thead><tbody></tbody></table></div>
    </div>
    <div class="story" id="storyStress"></div>
  </div>

  <!-- 08 -->
  <div class="section" id="s08">
    <div class="section-head"><h2><span class="num">08</span>Model Tiering Matrix</h2><span class="note">NB6 real composite rubric &middot; 3 dimensions</span></div>
    <div class="card two-col">
      <div class="stat-row" id="tierStats"></div>
      <div class="table-scroll"><table><thead><tr><th>Open governance flag</th></tr></thead><tbody id="flagRows"></tbody></table></div>
    </div>
    <div class="story" id="storyTiering"></div>
  </div>

  <!-- 09 -->
  <div class="section" id="s09">
    <div class="section-head"><h2><span class="num">09</span>BCBS 239 Data Governance</h2><span class="note">NB7 real data-quality gate + concentration + principle mapping</span></div>
    <div class="card two-col">
      <div class="stat-row" id="dqStats"></div>
      <div class="stat-row" id="concStats"></div>
    </div>
    <div class="card"><div class="table-scroll"><table id="bcbsTable"><thead><tr><th>Principle group</th><th>Status</th><th>Evidence</th></tr></thead><tbody></tbody></table></div></div>
    <div class="story" id="storyBcbs"></div>
  </div>

  <!-- 10 -->
  <div class="section" id="s10">
    <div class="section-head"><h2><span class="num">10</span>Regulatory Applicability &amp; Human Oversight</h2><span class="note">NB8 real inference logging + ASSUMPTION review band</span></div>
    <div class="card two-col">
      <div class="table-scroll"><table id="regTable"><thead><tr><th>Framework</th><th>Jurisdiction</th></tr></thead><tbody></tbody></table></div>
      <div class="stat-row" id="oversightStats"></div>
    </div>
    <div class="story" id="storyOversight"></div>
  </div>

  <!-- 11 -->
  <div class="section" id="s11">
    <div class="section-head"><h2><span class="num">11</span>Governance Sign-Off Readiness</h2><span class="note">Filter by sign-off status above &middot; NB9 real evidence-populated 4-tier dry-run</span></div>
    <div class="card">
      <div class="tier-bar" id="tierBar"></div>
      <div class="legend-inline" style="margin-top:10px;">
        <span><i style="background:var(--good)"></i>Pass (real evidence)</span>
        <span><i style="background:var(--warn)"></i>Conditional / open item</span>
        <span><i style="background:var(--critical)"></i>Outside pipeline scope (TBD)</span>
      </div>
      <div class="stat-row" style="margin-top:18px;" id="signoffStats"></div>
      <div class="table-scroll" style="margin-top:16px;"><table id="signoffTable"><thead><tr><th>Tier</th><th>Check</th><th>Verdict</th></tr></thead><tbody></tbody></table></div>
    </div>
    <div class="story" id="storySignoff"></div>
  </div>

  <!-- 12 NEW -->
  <div class="section" id="s12">
    <div class="section-head"><h2><span class="num">12</span>Confusion Matrix Composition</h2><span class="note">NB1 real out-of-fold outcome mix</span></div>
    <div class="card two-col">
      <div class="chart-wrap short"><canvas id="confusionDoughnut"></canvas></div>
      <div class="stat-row" id="confusionStats"></div>
    </div>
    <div class="story" id="storyConfusion"></div>
  </div>

  <!-- 13 NEW -->
  <div class="section" id="s13">
    <div class="section-head"><h2><span class="num">13</span>Model Tiering — Rubric Radar</h2><span class="note">NB6 real 3-dimension composite, radar view</span></div>
    <div class="card">
      <div class="chart-wrap"><canvas id="tierRadar"></canvas></div>
    </div>
    <div class="story" id="storyRadar"></div>
  </div>

  <!-- 14 NEW -->
  <div class="section" id="s14">
    <div class="section-head"><h2><span class="num">14</span>BCBS 239 Governance Mix</h2><span class="note">NB7 real principle-group status distribution</span></div>
    <div class="card">
      <div class="chart-wrap"><canvas id="bcbsPolar"></canvas></div>
    </div>
    <div class="story" id="storyBcbsMix"></div>
  </div>

  <!-- 15 NEW -->
  <div class="section" id="s15">
    <div class="section-head"><h2><span class="num">15</span>Drift History Trend</h2><span class="note">NB3 real append-only monitoring history</span></div>
    <div class="card">
      <div class="chart-wrap"><canvas id="driftLine"></canvas></div>
    </div>
    <div class="story" id="storyDriftTrend"></div>
  </div>

  <!-- 16 NEW -->
  <div class="section" id="s16">
    <div class="section-head"><h2><span class="num">16</span>Inference Log — Fraud Probability Distribution</h2><span class="note">NB8 real 551-record sample, 12 equal-width bins</span></div>
    <div class="card">
      <div class="chart-wrap"><canvas id="inferHist"></canvas></div>
    </div>
    <div class="story" id="storyInferHist"></div>
  </div>

  <!-- 17 NEW -->
  <div class="section" id="s17">
    <div class="section-head"><h2><span class="num">17</span>Reverse Stress Test</h2><span class="note">NB5 real fraud-rate multiplier required to breach cost targets</span></div>
    <div class="card">
      <div class="chart-wrap"><canvas id="reverseScatter"></canvas></div>
    </div>
    <div class="story" id="storyReverseScatter"></div>
  </div>

  <!-- SMART -->
  <div class="section"><div class="callout smart"><h4>Smart suggestions (auto-derived from real flags across NB1&ndash;NB9)</h4><ul id="smartList"></ul></div></div>

  <div class="footer">
    <span>Prepared by Nandagopal &middot; World-Class Edition v1.0 &middot; Real dataset, real models, real numbers — zero-fabrication policy</span>
    <span>Sources: NB1&ndash;NB9, {R['repo_root']} &middot; EUR/USD 1.1592 (ECB reference rate, 2026-09-11)</span>
  </div>

</div>

<script>
const DATA = {DATA_JSON};
const nb1 = DATA.nb1, nb2 = DATA.nb2, nb4 = DATA.nb4, nb5 = DATA.nb5, nb6 = DATA.nb6, nb7 = DATA.nb7, nb8 = DATA.nb8, nb9 = DATA.nb9;
const nb3_drift_history = DATA.nb3_drift_history, hist_bin_labels = DATA.hist_bin_labels, hist_bin_counts = DATA.hist_bin_counts;

function fmtPct(x, d=2) {{ return (x*100).toFixed(d) + "%"; }}
const EUR_TO_USD = 1.1592;
function fmtEur(x) {{ return "€" + x.toLocaleString("en-US", {{minimumFractionDigits:2, maximumFractionDigits:2}}) +
  "  (≈ $" + (x*EUR_TO_USD).toLocaleString("en-US", {{minimumFractionDigits:2, maximumFractionDigits:2}}) + ")"; }}
function fmtNum(x) {{ return x.toLocaleString("en-US"); }}
function r4(x) {{ return Number(x).toFixed(4); }}

const rootStyle = getComputedStyle(document.documentElement);
function cssVar(name) {{ return rootStyle.getPropertyValue(name).trim(); }}
const COLORS = {{
  accent: cssVar('--accent') || '#7C9CFF', accent2: cssVar('--accent2') || '#A78BFA', accent3: cssVar('--accent3') || '#34D6C7',
  good: cssVar('--good') || '#3DDC97', warn: cssVar('--warn') || '#F5B85A', critical: cssVar('--critical') || '#FF6B7A',
  ink: cssVar('--ink') || '#EAF0FF', inkSoft: cssVar('--ink-soft') || '#9FB0D6', border: cssVar('--surface-border') || 'rgba(255,255,255,0.09)',
}};
Chart.defaults.font.family = "Inter, sans-serif";
Chart.defaults.color = COLORS.inkSoft;
Chart.defaults.borderColor = COLORS.border;

// ---------------- Theme toggle ----------------
document.getElementById('themeToggle').addEventListener('click', () => {{
  const root = document.documentElement;
  const cur = root.getAttribute('data-theme');
  root.setAttribute('data-theme', cur === 'light' ? 'dark' : 'light');
}});

// ---------------- Reveal-on-scroll + animated counters ----------------
const _revealTargets = document.querySelectorAll('.section, .kpi');
const _io = new IntersectionObserver((entries) => {{
  entries.forEach(e => {{ if (e.isIntersecting) {{ e.target.classList.add('revealed'); _io.unobserve(e.target); }} }});
}}, {{ threshold: 0.08 }});
_revealTargets.forEach(t => _io.observe(t));

function animateCount(el, endVal, isCurrency, decimals) {{
  const dur = 900; const start = performance.now();
  function step(now) {{
    const t = Math.min(1, (now - start) / dur);
    const eased = 1 - Math.pow(1 - t, 3);
    const val = endVal * eased;
    el.textContent = isCurrency ? fmtEur(val) : (decimals ? val.toFixed(decimals) : Math.round(val).toLocaleString("en-US"));
    if (t < 1) requestAnimationFrame(step);
  }}
  requestAnimationFrame(step);
}}

// ---------------- KPI strip ----------------
const kpis = [
  {{label:"Champion Model", value: nb1.champion_name, sub: "vs. runner-up " + nb1.runner_up_name, cls:"", numeric:false}},
  {{label:"Champion CV PR-AUC", value: nb1.stage_b[nb1.champion_name].mean_pr_auc, sub: "5-fold stratified", cls:"good", numeric:true, decimals:4}},
  {{label:"Temporal PR-AUC", value: nb1.temporal_pr_auc, sub: "honest, held-out-in-time", cls:"warn", numeric:true, decimals:4}},
  {{label:"Savings vs. No Model", value: nb1.savings_vs_no_model, sub: "measured", cls:"good", numeric:true, currency:true}},
  {{label:"Model Tier (NB6)", value: "Tier " + nb6.model_tier, sub: nb6.tier_description, cls: nb6.model_tier<=1?"good":"warn", numeric:false}},
  {{label:"BCBS239 Gate", value: nb7.data_quality_gate.all_checks_passed ? "PASS" : "FAIL", sub: nb7.data_quality_gate.n_rows.toLocaleString()+" rows checked", cls: nb7.data_quality_gate.all_checks_passed?"good":"critical", numeric:false}},
  {{label:"Governance Readiness", value: (nb9.tier1_rows.length+nb9.tier2_rows.length+nb9.tier3_rows.length+nb9.tier4_rows.length)+" checks", sub: nb9.outside_pipeline_scope_count+" outside pipeline scope", cls:"", numeric:false}},
];
document.getElementById("kpiGrid").innerHTML = kpis.map((k,i) => `<div class="kpi ${{k.cls}}"><div class="label">${{k.label}}</div><div class="value" id="kpiVal${{i}}">${{k.numeric ? '0' : k.value}}</div><div class="sub">${{k.sub}}</div></div>`).join("");
kpis.forEach((k,i) => {{ if (k.numeric) {{ const el = document.getElementById('kpiVal'+i); setTimeout(() => animateCount(el, k.value, !!k.currency, k.decimals), 150 + i*80); }} }});
// The KPI cards above are created dynamically (after the reveal-observer was set up on the
// static .section elements), so they must be re-queried and observed here or they would sit
// at opacity:0 forever. Real bug found+fixed during NB13 verification (2026-09-13).
document.querySelectorAll('.kpi').forEach(t => _io.observe(t));

// ---------------- 01 Stage A ----------------
const stageA = nb1.stage_a;
let stageAChart;
function renderStageA(highlightModel) {{
  const labels = stageA.map(d => d.model);
  const values = stageA.map(d => d.val_pr_auc);
  const bg = stageA.map((d,i) => (highlightModel==="all"||d.model===highlightModel) ? (i<2?COLORS.accent:COLORS.accent2) : COLORS.border);
  if (stageAChart) stageAChart.destroy();
  stageAChart = new Chart(document.getElementById('stageAChart'), {{ type:'bar',
    data: {{ labels, datasets:[{{ label:'Validation PR-AUC', data: values, backgroundColor: bg, borderRadius:8, maxBarThickness:34 }}] }},
    options: {{ indexAxis:'y', responsive:true, maintainAspectRatio:false, animation: {{duration: 700}},
      plugins: {{ legend:{{display:false}}, title:{{display:true, text:'Stage A Screening — Validation PR-AUC', color:COLORS.ink, font:{{size:13,weight:600}}}} }},
      scales: {{ x:{{min:0,max:1,grid:{{color:COLORS.border}}}}, y:{{grid:{{display:false}}}} }} }} }});
  const tb = document.querySelector('#stageATable tbody');
  tb.innerHTML = stageA.map((d,i) => `<tr class="${{(highlightModel!=='all'&&d.model!==highlightModel)?'dim':''}}"><td class="mono">${{i+1}}</td><td>${{d.model}}</td><td class="mono">${{r4(d.val_pr_auc)}}</td><td>${{i<2?'<span class="pill good">Advances</span>':'<span class="pill neutral">Screened out</span>'}}</td></tr>`).join("");
}}
document.getElementById('modelFilter').innerHTML = ['all', ...stageA.map(d=>d.model)].map((m,i) => `<button class="chip ${{i===0?'active':''}}" data-model="${{m}}">${{m==='all'?'All Models':m}}</button>`).join("");
document.querySelectorAll('#modelFilter .chip').forEach(btn => btn.addEventListener('click', () => {{ document.querySelectorAll('#modelFilter .chip').forEach(b=>b.classList.remove('active')); btn.classList.add('active'); renderStageA(btn.dataset.model); }}));
renderStageA('all');
document.getElementById('storyStageA').innerHTML = `<b>The story:</b> ${{stageA.length}} real candidates were screened; ${{stageA[0].model}} (${{r4(stageA[0].val_pr_auc)}}) and ${{stageA[1].model}} (${{r4(stageA[1].val_pr_auc)}}) led and advanced to Stage B. ${{stageA[stageA.length-1].model}} trailed at ${{r4(stageA[stageA.length-1].val_pr_auc)}}, the widest real gap in the screen.`;

// ---------------- 02 CV vs Temporal ----------------
const champ = nb1.champion_name, runner = nb1.runner_up_name;
const sb = nb1.stage_b;
new Chart(document.getElementById('cvChart'), {{ type:'bar',
  data: {{ labels:['Fold 1','Fold 2','Fold 3','Fold 4','Fold 5'], datasets:[
    {{ label:champ, data: sb[champ].fold_pr_auc, backgroundColor: COLORS.accent, borderRadius:6 }},
    {{ label:runner, data: sb[runner].fold_pr_auc, backgroundColor: COLORS.accent2, borderRadius:6 }} ] }},
  options: {{ responsive:true, maintainAspectRatio:false, animation:{{duration:700}},
    plugins: {{ legend:{{position:'bottom', labels:{{boxWidth:10, font:{{size:11}}}}}}, title:{{display:true, text:'5-Fold CV — Champion vs. Runner-up', color:COLORS.ink, font:{{size:13,weight:600}}}} }},
    scales: {{ y:{{min:0.7,max:0.95,grid:{{color:COLORS.border}}}}, x:{{grid:{{display:false}}}} }} }} }});
new Chart(document.getElementById('temporalChart'), {{ type:'bar',
  data: {{ labels:['5-fold CV (shuffled)','Temporal split (honest)'], datasets:[{{ data:[sb[champ].mean_pr_auc, nb1.temporal_pr_auc], backgroundColor:[COLORS.accent3,COLORS.warn], borderRadius:8, maxBarThickness:60 }}] }},
  options: {{ responsive:true, maintainAspectRatio:false, animation:{{duration:700}}, plugins:{{legend:{{display:false}}, title:{{display:true,text:'CV vs. Temporal-Split PR-AUC', color:COLORS.ink, font:{{size:13,weight:600}}}}}}, scales:{{y:{{min:0,max:1,grid:{{color:COLORS.border}}}}, x:{{grid:{{display:false}}}}}} }} }});
document.getElementById('storyValidation').innerHTML = `<b>The story:</b> Shuffled 5-fold CV reports ${{r4(sb[champ].mean_pr_auc)}} PR-AUC, but the honest temporal split — training on earlier transactions, testing on strictly later ones — drops to ${{r4(nb1.temporal_pr_auc)}}. That real ${{((sb[champ].mean_pr_auc-nb1.temporal_pr_auc)*100).toFixed(1)}}-point gap is disclosed, not hidden.`;

// ---------------- 03 Financial Impact ----------------
const finRows = nb1.financial_impact;
const scenarios = [
  {{ key:'no_model', label:'No Model', cost: finRows[0]["Total Cost (EUR)"], fn: nb1.dataset.fraud_count, fp: 0 }},
  {{ key:'naive', label:'Naive 0.5', cost: finRows[1]["Total Cost (EUR)"], fn: null, fp: null }},
  {{ key:'optimal', label:'Cost-Optimal', cost: finRows[2]["Total Cost (EUR)"], fn: nb1.confusion_matrix[1][0], fp: nb1.confusion_matrix[0][1] }},
];
let finChart;
function renderFin(selected) {{
  const bg = scenarios.map(s => s.key===selected ? COLORS.accent : COLORS.border);
  if (finChart) finChart.destroy();
  finChart = new Chart(document.getElementById('finChart'), {{ type:'bar',
    data: {{ labels: scenarios.map(s=>s.label), datasets:[{{ label:'Total Cost (EUR, USD in tooltip)', data: scenarios.map(s=>s.cost), backgroundColor: bg, borderRadius:10, maxBarThickness:70 }}] }},
    options: {{ responsive:true, maintainAspectRatio:false, animation:{{duration:700}},
      plugins: {{ legend:{{display:false}}, title:{{display:true, text:'Real Total Cost by Scenario (EUR — hover for USD)', color:COLORS.ink, font:{{size:13,weight:600}}}}, tooltip:{{callbacks:{{label:(ctx)=>fmtEur(ctx.parsed.y)}}}} }},
      scales: {{ y:{{grid:{{color:COLORS.border}}, ticks:{{callback:(v)=>'€'+(v/1000)+'k'}}}}, x:{{grid:{{display:false}}}} }} }} }});
  const s = scenarios.find(x=>x.key===selected);
  const savingsVsNoModel = scenarios[0].cost - s.cost;
  document.getElementById('finStats').innerHTML = `
    <div class="stat"><div class="k">Total Cost</div><div class="v">${{fmtEur(s.cost)}}</div></div>
    <div class="stat"><div class="k">False Negatives</div><div class="v">${{s.fn ?? '—'}}</div></div>
    <div class="stat"><div class="k">False Positives</div><div class="v">${{s.fp ?? '—'}}</div></div>
    <div class="stat"><div class="k">Savings vs. No Model</div><div class="v" style="color:${{COLORS.good}}">${{fmtEur(savingsVsNoModel)}}</div></div>`;
  document.getElementById('storyFinancial').innerHTML = `<b>The story:</b> Choosing "${{s.label}}" costs ${{fmtEur(s.cost)}} in real measured losses — ${{fmtEur(savingsVsNoModel)}} better than doing nothing at all. The cost-optimal threshold beats every alternative scenario shown here.`;
}}
document.getElementById('scenarioFilter').innerHTML = scenarios.map((s,i) => `<button class="chip ${{i===2?'active':''}}" data-scenario="${{s.key}}">${{s.label}}</button>`).join("");
document.querySelectorAll('#scenarioFilter .chip').forEach(btn => btn.addEventListener('click', () => {{ document.querySelectorAll('#scenarioFilter .chip').forEach(b=>b.classList.remove('active')); btn.classList.add('active'); renderFin(btn.dataset.scenario); }}));
renderFin('optimal');

const bc = nb1.benchmark_check;
document.getElementById('benchmarkRows').innerHTML = `
  <tr><td>Precision</td><td class="mono">${{fmtPct(bc.your_precision,2)}}</td><td class="mono">${{fmtPct(bc.reference_precision,2)}}</td></tr>
  <tr><td>Recall</td><td class="mono">${{fmtPct(bc.your_recall,2)}}</td><td class="mono">${{fmtPct(bc.reference_recall,2)}}</td></tr>`;
document.getElementById('benchmarkCallout').innerHTML = `<h4>Investigate flag: ${{bc.investigate_flag}}</h4><ul><li>Precision gap of ${{fmtPct(bc.precision_gap,2)}} vs. an external reference exceeds tolerance — flagged for human investigation before external sign-off. ${{bc.note}}</li></ul>`;

// ---------------- 04 Governance Gates & Drift ----------------
const g1 = nb2.gate1_structural_checks;
const g1Pass = Object.values(g1).filter(Boolean).length;
document.getElementById('gateStats').innerHTML = `
  <div class="stat"><div class="k">Gate 1 — Structural</div><div class="v"><span class="pill ${{nb2.gate1_all_passed?'good':'critical'}}">${{g1Pass}}/${{Object.keys(g1).length}} checks passed</span></div></div>
  <div class="stat"><div class="k">Gate 2 — CV Stability</div><div class="v"><span class="pill ${{nb2.gate2_cv_stability_ok?'good':'critical'}}">${{nb2.gate2_cv_stability_ok?'OK':'FAIL'}}</span></div></div>
  <div class="stat"><div class="k">Any Drift Alert</div><div class="v"><span class="pill ${{nb2.drift_monitoring.any_alert?'warn':'good'}}">${{nb2.drift_monitoring.any_alert}}</span></div></div>
  <div class="stat"><div class="k">Score KS statistic</div><div class="v">${{nb2.drift_monitoring.score_ks_statistic.toFixed(4)}}</div></div>`;
const psiEntries = Object.entries(nb2.drift_monitoring.feature_psi);
const psiWorstEntry = psiEntries.reduce((a,b) => a[1]>b[1]?a:b);
document.getElementById('psiVal').textContent = psiWorstEntry[1].toFixed(4);
document.getElementById('psiMarker').style.left = Math.min(100, (psiWorstEntry[1]/1.5)*100) + '%';
document.getElementById('storyDrift').innerHTML = `<b>The story:</b> Gate 1 passed ${{g1Pass}} of ${{Object.keys(g1).length}} real structural checks. Feature ${{psiWorstEntry[0]}} shows the worst real drift (PSI ${{psiWorstEntry[1].toFixed(4)}}) between the early and late windows — any_alert is ${{nb2.drift_monitoring.any_alert}}.`;

// ---------------- 05 Adversarial ----------------
const adv = nb2.adversarial_robustness.perturbation_sensitivity;
let advTypeFilter = 'all';
function renderAdv() {{
  const tb = document.querySelector('#advTable tbody');
  tb.innerHTML = adv.filter(d => advTypeFilter==='all' || d.perturbation_type===advTypeFilter).map(d => `
    <tr><td>${{d.perturbation_type}}</td><td class="mono">${{d.perturbation_value}}</td><td class="mono">${{d.n_fraud_originally_caught}}</td>
    <td class="mono">${{d.n_fraud_still_caught_after}}</td><td class="mono"><span class="pill ${{d.fraud_evasion_rate>0?'warn':'good'}}">${{fmtPct(d.fraud_evasion_rate,2)}}</span></td></tr>`).join("");
}}
document.getElementById('advFilter').innerHTML = ['all','amount','time'].map((t,i) => `<button class="chip ${{i===0?'active':''}}" data-type="${{t}}">${{t==='all'?'All Types':t}}</button>`).join("");
document.querySelectorAll('#advFilter .chip').forEach(btn => btn.addEventListener('click', () => {{ document.querySelectorAll('#advFilter .chip').forEach(b=>b.classList.remove('active')); btn.classList.add('active'); advTypeFilter = btn.dataset.type; renderAdv(); }}));
renderAdv();
const bnd = nb2.adversarial_robustness.boundary_search;
document.getElementById('boundaryStats').innerHTML = `
  <div class="stat"><div class="k">Fraud cases tested</div><div class="v">${{bnd.n_fraud_cases_tested}}</div></div>
  <div class="stat"><div class="k">Evadable within budget</div><div class="v">${{bnd.n_evadable_within_budget}}</div></div>
  <div class="stat"><div class="k">Evasion rate (budgeted)</div><div class="v">${{fmtPct(bnd.evasion_rate_within_budget,2)}}</div></div>
  <div class="stat"><div class="k">Median Δ for evasion</div><div class="v">${{fmtPct(bnd.median_amount_change_pct_for_evasion,0)}}</div></div>`;
document.getElementById('storyAdversarial').innerHTML = `<b>The story:</b> Within a ${{fmtPct(bnd.max_amount_change_pct_budget,0)}} amount-change budget, ${{bnd.n_evadable_within_budget}} of ${{bnd.n_fraud_cases_tested}} real fraud cases could be evaded (${{fmtPct(bnd.evasion_rate_within_budget,2)}}) — evaluated in-sample, so treat ${{fmtPct(nb1.real_recall,2)}} held-out recall as the honest ceiling.`;

// ---------------- 06 Deployment ----------------
const lat = nb4.latency_sla_ms.client_round_trip;
document.getElementById('latStats').innerHTML = `
  <div class="stat"><div class="k">Mean Latency (client)</div><div class="v">${{lat.mean.toFixed(2)}} ms</div></div>
  <div class="stat"><div class="k">p50 Latency</div><div class="v">${{lat.p50.toFixed(2)}} ms</div></div>
  <div class="stat"><div class="k">p95 Latency</div><div class="v">${{lat.p95.toFixed(2)}} ms</div></div>
  <div class="stat"><div class="k">p99 Latency</div><div class="v">${{lat.p99.toFixed(2)}} ms</div></div>`;
const tsc = nb4.training_serving_consistency;
document.getElementById('consistencyStats').innerHTML = `
  <div class="stat"><div class="k">Rows tested</div><div class="v">${{tsc.n_rows_tested}}</div></div>
  <div class="stat"><div class="k">Max abs diff</div><div class="v">${{tsc.max_abs_diff}}</div></div>
  <div class="stat"><div class="k">Consistency</div><div class="v"><span class="pill ${{tsc.passed?'good':'critical'}}">${{tsc.passed?'PASS':'FAIL'}}</span></div></div>
  <div class="stat"><div class="k">Health check</div><div class="v"><span class="pill ${{nb4.health_check_passed?'good':'critical'}}">${{nb4.health_check_passed?'PASS':'FAIL'}}</span></div></div>`;
document.getElementById('storyDeployment').innerHTML = `<b>The story:</b> The real local API answers in ${{lat.mean.toFixed(2)}} ms on average (p99 ${{lat.p99.toFixed(2)}} ms), and training/serving parity is exact — max abs diff ${{tsc.max_abs_diff}} across ${{tsc.n_rows_tested}} real rows.`;

// ---------------- 07 Stress Testing ----------------
const ladder = nb5.scenario_ladder;
let stressChart;
function renderStress(sel) {{
  const bg = ladder.map(s => (sel==='all'||s.tier_name===sel) ? COLORS.accent : COLORS.border);
  if (stressChart) stressChart.destroy();
  stressChart = new Chart(document.getElementById('stressChart'), {{ type:'bar',
    data: {{ labels: ladder.map(s=>s.tier_name), datasets:[{{ label:'Projected Loss (EUR)', data: ladder.map(s=>s.projected_fraud_loss_usd_or_eur), backgroundColor:bg, borderRadius:10, maxBarThickness:80 }}] }},
    options: {{ responsive:true, maintainAspectRatio:false, animation:{{duration:700}},
      plugins: {{ legend:{{display:false}}, title:{{display:true, text:'3-Tier Stress Scenario Ladder (disclosed ASSUMPTION multipliers)', color:COLORS.ink, font:{{size:13,weight:600}}}}, tooltip:{{callbacks:{{label:(ctx)=>fmtEur(ctx.parsed.y)}}}} }},
      scales: {{ y:{{grid:{{color:COLORS.border}}, ticks:{{callback:(v)=>'€'+(v/1000)+'k'}}}}, x:{{grid:{{display:false}}}} }} }} }});
}}
document.getElementById('stressFilter').innerHTML = ['all', ...ladder.map(s=>s.tier_name)].map((t,i) => `<button class="chip ${{i===0?'active':''}}" data-tier="${{t}}">${{t==='all'?'All Tiers':t}}</button>`).join("");
document.querySelectorAll('#stressFilter .chip').forEach(btn => btn.addEventListener('click', () => {{ document.querySelectorAll('#stressFilter .chip').forEach(b=>b.classList.remove('active')); btn.classList.add('active'); renderStress(btn.dataset.tier); }}));
renderStress('all');
document.querySelector('#reverseStressTable tbody').innerHTML = nb5.reverse_stress_test.results.map(r => `
  <tr><td>${{r.target_multiple_of_current_cost}}&times; current cost</td><td class="mono">${{r.required_fraud_rate_multiplier.toFixed(2)}}&times;</td><td class="mono">${{fmtPct(r.required_fraud_rate,3)}}</td></tr>`).join("");
const wg = nb5.grid_sweep.worst_combination;
document.getElementById('storyStress').innerHTML = `<b>The story:</b> The worst of ${{nb5.grid_sweep.n_combinations}} real swept combinations — volume &times;${{wg.volume_multiplier}}, fraud-rate &times;${{wg.fraud_rate_multiplier}} — projects a loss of ${{fmtEur(wg.projected_loss_eur)}}, ${{(wg.projected_loss_eur/ladder[0].projected_fraud_loss_usd_or_eur).toFixed(1)}}&times; today's real baseline.`;

// ---------------- 08 Model Tiering ----------------
const rub = nb6.rubric;
document.getElementById('tierStats').innerHTML = `
  <div class="stat"><div class="k">Financial Exposure</div><div class="v">${{rub.financial_exposure.score}}/3</div></div>
  <div class="stat"><div class="k">Regulatory Scrutiny</div><div class="v">${{rub.regulatory_scrutiny.score}}/3</div></div>
  <div class="stat"><div class="k">Customer Impact</div><div class="v">${{rub.customer_impact.score}}/3</div></div>
  <div class="stat"><div class="k">Composite Score</div><div class="v">${{nb6.composite_score}}/9 &rarr; Tier ${{nb6.model_tier}}</div></div>`;
document.getElementById('flagRows').innerHTML = rub.regulatory_scrutiny.open_flags.map(f => `<tr><td style="font-size:12.5px;">${{f}}</td></tr>`).join("");
document.getElementById('storyTiering').innerHTML = `<b>The story:</b> A composite score of ${{nb6.composite_score}}/9 places the model at Tier ${{nb6.model_tier}} (${{nb6.tier_description}}) — driven mainly by a ${{rub.financial_exposure.stress_multiple.toFixed(1)}}&times; worst-case financial exposure and ${{rub.regulatory_scrutiny.open_flags.length}} open regulatory flags.`;

// ---------------- 09 BCBS 239 ----------------
const dq = nb7.data_quality_gate;
const dqPass = Object.values(dq.checks).filter(Boolean).length;
document.getElementById('dqStats').innerHTML = `
  <div class="stat"><div class="k">Data-quality checks</div><div class="v"><span class="pill ${{dq.all_checks_passed?'good':'critical'}}">${{dqPass}}/${{Object.keys(dq.checks).length}} passed</span></div></div>
  <div class="stat"><div class="k">Rows checked</div><div class="v">${{fmtNum(dq.n_rows)}}</div></div>
  <div class="stat"><div class="k">Fresh duplicate rows</div><div class="v">${{dq.duplicate_row_count_fresh}} (${{fmtPct(dq.duplicate_row_rate_fresh,2)}})</div></div>`;
const cc = nb7.concentration_summary;
document.getElementById('concStats').innerHTML = `
  <div class="stat"><div class="k">Segments analyzed</div><div class="v">${{cc.n_segments}}</div></div>
  <div class="stat"><div class="k">Worst FP rate segment</div><div class="v">${{fmtPct(cc.max_false_positive_rate,2)}}</div></div>
  <div class="stat"><div class="k">Worst FN rate segment</div><div class="v">${{fmtPct(cc.max_false_negative_rate,2)}}</div></div>`;
document.querySelector('#bcbsTable tbody').innerHTML = nb7.bcbs239_mapping.map(m => `
  <tr><td>${{m.principle_group}}</td><td><span class="pill ${{m.status==='evidenced'?'good':'warn'}}">${{m.status.replace('_',' ')}}</span></td><td style="font-size:12.5px;">${{m.evidence}}</td></tr>`).join("");
document.getElementById('storyBcbs').innerHTML = `<b>The story:</b> ${{dqPass}} of ${{Object.keys(dq.checks).length}} real data-quality checks passed over ${{fmtNum(dq.n_rows)}} rows; ${{nb7.bcbs239_mapping.filter(m=>m.status!=='evidenced').length}} of ${{nb7.bcbs239_mapping.length}} BCBS 239 principle groups remain limitation-disclosed, not evidenced.`;

// ---------------- 10 Regulatory & Oversight ----------------
document.querySelector('#regTable tbody').innerHTML = nb8.regulatory_applicability.map(f => `<tr><td>${{f.framework}}</td><td><span class="pill neutral">${{f.jurisdiction}}</span></td></tr>`).join("");
const ho = nb8.human_oversight;
document.getElementById('oversightStats').innerHTML = `
  <div class="stat"><div class="k">Review band (±${{ho.band_half_width_ASSUMPTION}})</div><div class="v">${{ho.band_lo.toFixed(4)}}&ndash;${{ho.band_hi.toFixed(4)}}</div></div>
  <div class="stat"><div class="k">Transactions in band</div><div class="v">${{ho.n_in_band}} / ${{fmtNum(ho.n_total)}}</div></div>
  <div class="stat"><div class="k">Fraud in band</div><div class="v">${{ho.n_fraud_in_band}}</div></div>
  <div class="stat"><div class="k">Legit in band</div><div class="v">${{ho.n_legit_in_band}}</div></div>`;
document.getElementById('storyOversight').innerHTML = `<b>The story:</b> ${{nb8.regulatory_applicability.length}} real regulatory frameworks apply. A disclosed ±${{ho.band_half_width_ASSUMPTION}} review band around the real threshold routes ${{ho.n_in_band}} of ${{fmtNum(ho.n_total)}} transactions for manual human review — ${{ho.n_fraud_in_band}} of them real fraud.`;

// ---------------- 11 Governance Sign-Off ----------------
function tierCounts(rows) {{
  let pass=0, cond=0, tbd=0;
  rows.forEach(r => {{ const v=r[2]; if (v.startsWith('Pass')) pass++; else if (v.startsWith('TBD')) tbd++; else cond++; }});
  return {{pass, cond, tbd, total: rows.length}};
}}
const t1=tierCounts(nb9.tier1_rows), t2=tierCounts(nb9.tier2_rows), t3=tierCounts(nb9.tier3_rows), t4=tierCounts(nb9.tier4_rows);
const allT = t1.total+t2.total+t3.total+t4.total, allPass = t1.pass+t2.pass+t3.pass+t4.pass, allCond = t1.cond+t2.cond+t3.cond+t4.cond, allTbd = t1.tbd+t2.tbd+t3.tbd+t4.tbd;
document.getElementById('tierBar').innerHTML = `
  <div style="width:${{allPass/allT*100}}%;background:${{COLORS.good}}">${{allPass}} Pass</div>
  <div style="width:${{allCond/allT*100}}%;background:${{COLORS.warn}}">${{allCond}} Conditional</div>
  <div style="width:${{allTbd/allT*100}}%;background:${{COLORS.critical}}">${{allTbd}} TBD</div>`;
document.getElementById('signoffStats').innerHTML = `
  <div class="stat"><div class="k">Tier 1 — Technical Lead</div><div class="v">${{t1.pass}}/${{t1.total}} Pass</div></div>
  <div class="stat"><div class="k">Tier 2 — Model Risk Manager</div><div class="v">${{t2.pass}}/${{t2.total}} Pass</div></div>
  <div class="stat"><div class="k">Tier 3 — CCO</div><div class="v">${{t3.pass}}/${{t3.total}} Pass</div></div>
  <div class="stat"><div class="k">Tier 4 — Business Owner</div><div class="v">${{t4.pass}}/${{t4.total}} Pass</div></div>`;
const ALL_ROWS = [
  ...nb9.tier1_rows.map(r => ({{tier:'Tier 1', check:r[0], verdict:r[2]}})),
  ...nb9.tier2_rows.map(r => ({{tier:'Tier 2', check:r[0], verdict:r[2]}})),
  ...nb9.tier3_rows.map(r => ({{tier:'Tier 3', check:r[0], verdict:r[2]}})),
  ...nb9.tier4_rows.map(r => ({{tier:'Tier 4', check:r[0], verdict:r[2]}})),
];
let statusFilter = 'all';
function renderSignoffTable() {{
  const tb = document.querySelector('#signoffTable tbody');
  const rows = ALL_ROWS.filter(r => statusFilter==='all' ||
    (statusFilter==='Pass' && r.verdict.startsWith('Pass')) ||
    (statusFilter==='TBD' && r.verdict.startsWith('TBD')) ||
    (statusFilter==='Conditional' && !r.verdict.startsWith('Pass') && !r.verdict.startsWith('TBD')));
  tb.innerHTML = rows.map(r => `<tr><td>${{r.tier}}</td><td style="font-size:12.5px;">${{r.check}}</td><td><span class="pill ${{r.verdict.startsWith('Pass')?'good':(r.verdict.startsWith('TBD')?'critical':'warn')}}">${{r.verdict.split(' -- ')[0]}}</span></td></tr>`).join("");
}}
document.getElementById('statusFilter').innerHTML = ['all','Pass','Conditional','TBD'].map((s,i) => `<button class="chip ${{i===0?'active':''}}" data-status="${{s}}">${{s==='all'?'All':s}}</button>`).join("");
document.querySelectorAll('#statusFilter .chip').forEach(btn => btn.addEventListener('click', () => {{ document.querySelectorAll('#statusFilter .chip').forEach(b=>b.classList.remove('active')); btn.classList.add('active'); statusFilter = btn.dataset.status; renderSignoffTable(); }}));
renderSignoffTable();
document.getElementById('storySignoff').innerHTML = `<b>The story:</b> ${{allPass}} of ${{allT}} real governance checks pass on evidence alone; ${{allTbd}} are marked outside this pipeline's scope (real organizational facts, not model gaps) — no signature has been fabricated for any of them.`;

// ---------------- 12 NEW Confusion Matrix Doughnut ----------------
const cm = nb1.confusion_matrix; // [[TN,FP],[FN,TP]]
const cmVals = {{ TN: cm[0][0], FP: cm[0][1], FN: cm[1][0], TP: cm[1][1] }};
new Chart(document.getElementById('confusionDoughnut'), {{ type: 'doughnut',
  data: {{ labels: ['True Negative','False Positive','False Negative','True Positive'],
    datasets: [{{ data: [cmVals.TN, cmVals.FP, cmVals.FN, cmVals.TP], backgroundColor: [COLORS.accent2, COLORS.warn, COLORS.critical, COLORS.good], borderWidth: 0 }}] }},
  options: {{ responsive:true, maintainAspectRatio:false, animation:{{duration:800}}, cutout:'62%',
    plugins: {{ legend:{{position:'bottom', labels:{{boxWidth:10,font:{{size:11}}}}}}, title:{{display:true, text:'Real Out-of-Fold Confusion Matrix', color:COLORS.ink, font:{{size:13,weight:600}}}} }} }} }});
document.getElementById('confusionStats').innerHTML = `
  <div class="stat"><div class="k">True Positive (caught fraud)</div><div class="v">${{fmtNum(cmVals.TP)}}</div></div>
  <div class="stat"><div class="k">False Negative (missed fraud)</div><div class="v">${{fmtNum(cmVals.FN)}}</div></div>
  <div class="stat"><div class="k">False Positive (false alarm)</div><div class="v">${{fmtNum(cmVals.FP)}}</div></div>
  <div class="stat"><div class="k">True Negative (correct legit)</div><div class="v">${{fmtNum(cmVals.TN)}}</div></div>`;
document.getElementById('storyConfusion').innerHTML = `<b>The story:</b> Of ${{fmtNum(cmVals.TN+cmVals.FP+cmVals.FN+cmVals.TP)}} real transactions, the model correctly caught ${{fmtNum(cmVals.TP)}} frauds and missed ${{fmtNum(cmVals.FN)}}, while raising ${{fmtNum(cmVals.FP)}} false alarms against ${{fmtNum(cmVals.TN)}} correctly-cleared legitimate transactions.`;

// ---------------- 13 NEW Tiering Radar ----------------
new Chart(document.getElementById('tierRadar'), {{ type: 'radar',
  data: {{ labels: ['Financial Exposure','Regulatory Scrutiny','Customer Impact'],
    datasets: [{{ label: 'Real rubric score (/3)', data: [rub.financial_exposure.score, rub.regulatory_scrutiny.score, rub.customer_impact.score],
      backgroundColor: 'rgba(124,156,255,0.25)', borderColor: COLORS.accent, pointBackgroundColor: COLORS.accent, borderWidth: 2 }}] }},
  options: {{ responsive:true, maintainAspectRatio:false, animation:{{duration:800}},
    plugins: {{ legend:{{display:false}}, title:{{display:true, text:`Composite ${{nb6.composite_score}}/9 -- Tier ${{nb6.model_tier}}`, color:COLORS.ink, font:{{size:13,weight:600}}}} }},
    scales: {{ r: {{ min:0, max:3, ticks:{{stepSize:1, backdropColor:'transparent'}}, grid:{{color:COLORS.border}}, angleLines:{{color:COLORS.border}} }} }} }} }});
document.getElementById('storyRadar').innerHTML = `<b>The story:</b> Financial Exposure scores ${{rub.financial_exposure.score}}/3, Regulatory Scrutiny ${{rub.regulatory_scrutiny.score}}/3, and Customer Impact ${{rub.customer_impact.score}}/3 -- the widest dimension is what pushed the model into Tier ${{nb6.model_tier}} rather than a lighter-touch tier.`;

// ---------------- 14 NEW BCBS239 Polar Area ----------------
const bcbsStatusCounts = nb7.bcbs239_mapping.reduce((acc,m) => {{ acc[m.status] = (acc[m.status]||0)+1; return acc; }}, {{}});
new Chart(document.getElementById('bcbsPolar'), {{ type: 'polarArea',
  data: {{ labels: Object.keys(bcbsStatusCounts).map(s=>s.replace('_',' ')), datasets: [{{ data: Object.values(bcbsStatusCounts),
    backgroundColor: [COLORS.good+'CC', COLORS.warn+'CC', COLORS.critical+'CC'] }}] }},
  options: {{ responsive:true, maintainAspectRatio:false, animation:{{duration:800}},
    plugins: {{ legend:{{position:'bottom'}}, title:{{display:true, text:'BCBS 239 Principle Groups by Status', color:COLORS.ink, font:{{size:13,weight:600}}}} }},
    scales: {{ r: {{ grid:{{color:COLORS.border}}, ticks:{{backdropColor:'transparent'}} }} }} }} }});
document.getElementById('storyBcbsMix').innerHTML = `<b>The story:</b> Of ${{nb7.bcbs239_mapping.length}} real principle groups mapped, ${{bcbsStatusCounts['evidenced']||0}} are fully evidenced and ${{bcbsStatusCounts['limitation_disclosed']||0}} remain limitation-disclosed -- an honest split, not a rounded-up compliance claim.`;

// ---------------- 15 NEW Drift History Line ----------------
if (nb3_drift_history.length > 0) {{
  new Chart(document.getElementById('driftLine'), {{ type: 'line',
    data: {{ labels: nb3_drift_history.map((_,i) => 'Run ' + (i+1)),
      datasets: [
        {{ label: 'PR-AUC (baseline window)', data: nb3_drift_history.map(d=>d.pr_auc_baseline), borderColor: COLORS.accent, backgroundColor: 'transparent', tension:0.3 }},
        {{ label: 'PR-AUC (current window)', data: nb3_drift_history.map(d=>d.pr_auc_current), borderColor: COLORS.accent3, backgroundColor: 'transparent', tension:0.3 }},
      ] }},
    options: {{ responsive:true, maintainAspectRatio:false, animation:{{duration:800}},
      plugins: {{ legend:{{position:'bottom'}}, title:{{display:true, text: nb3_drift_history.length + ' Real Monitoring Run(s) -- PR-AUC Baseline vs. Current', color:COLORS.ink, font:{{size:13,weight:600}}}} }},
      scales: {{ y:{{min:0,max:1.05,grid:{{color:COLORS.border}}}}, x:{{grid:{{display:false}}}} }} }} }});
  document.getElementById('storyDriftTrend').innerHTML = `<b>The story:</b> Across ${{nb3_drift_history.length}} real append-only monitoring run(s), PR-AUC drop has stayed at ${{nb3_drift_history[nb3_drift_history.length-1].pr_auc_drop.toFixed(4)}} -- no performance degradation detected yet, though the window is still short.`;
}} else {{
  document.getElementById('storyDriftTrend').innerHTML = `<b>The story:</b> No real monitoring runs are on file yet -- rerun NB3 against a new data batch to populate this trend.`;
}}

// ---------------- 16 NEW Inference Log Histogram ----------------
new Chart(document.getElementById('inferHist'), {{ type: 'bar',
  data: {{ labels: hist_bin_labels, datasets: [{{ label: 'Real inference-log records', data: hist_bin_counts, backgroundColor: COLORS.accent2, borderRadius: 4 }}] }},
  options: {{ responsive:true, maintainAspectRatio:false, animation:{{duration:800}},
    plugins: {{ legend:{{display:false}}, title:{{display:true, text:'Fraud-Probability Distribution, 551 Real Records', color:COLORS.ink, font:{{size:13,weight:600}}}} }},
    scales: {{ y:{{grid:{{color:COLORS.border}}}}, x:{{grid:{{display:false}}, ticks:{{maxRotation:60, minRotation:60, font:{{size:9}}}}}} }} }} }});
const _histPeakIdx = hist_bin_counts.indexOf(Math.max(...hist_bin_counts));
document.getElementById('storyInferHist').innerHTML = `<b>The story:</b> Most of the 551 real sampled inference records cluster in the ${{hist_bin_labels[_histPeakIdx]}} probability bin (${{hist_bin_counts[_histPeakIdx]}} records) -- consistent with a heavily imbalanced real fraud rate where most transactions score well below the decision threshold.`;

// ---------------- 17 NEW Reverse Stress Scatter ----------------
const revResults = nb5.reverse_stress_test.results;
new Chart(document.getElementById('reverseScatter'), {{ type: 'line',
  data: {{ labels: revResults.map(r => r.target_multiple_of_current_cost + 'x cost'),
    datasets: [{{ label: 'Required fraud-rate multiplier', data: revResults.map(r => r.required_fraud_rate_multiplier),
      borderColor: COLORS.warn, backgroundColor: COLORS.warn + '33', pointRadius: 6, pointBackgroundColor: COLORS.warn, tension: 0.25, fill: true }}] }},
  options: {{ responsive:true, maintainAspectRatio:false, animation:{{duration:800}},
    plugins: {{ legend:{{display:false}}, title:{{display:true, text:'Real Fraud-Rate Multiplier Required to Breach Each Cost Target', color:COLORS.ink, font:{{size:13,weight:600}}}} }},
    scales: {{ y:{{grid:{{color:COLORS.border}}, title:{{display:true,text:'Required multiplier (x)'}}}}, x:{{grid:{{display:false}}}} }} }} }});
document.getElementById('storyReverseScatter').innerHTML = `<b>The story:</b> Doubling today's real cost needs only a ${{revResults[0].required_fraud_rate_multiplier.toFixed(2)}}&times; real fraud-rate increase; reaching ${{revResults[revResults.length-1].target_multiple_of_current_cost}}&times; needs ${{revResults[revResults.length-1].required_fraud_rate_multiplier.toFixed(2)}}&times; -- the curve shows how fast exposure compounds.`;

// ---------------- Smart Suggestions ----------------
const smart = [];
smart.push(`Adopt the cost-optimal threshold (${{r4(nb1.threshold_result.threshold)}}) over the default 0.5 — measured to save an additional ${{fmtEur(nb1.savings_vs_naive_05)}}.`);
if (bc.investigate_flag) smart.push(`Investigate the benchmark precision gap (${{fmtPct(bc.precision_gap,2)}}) before external sign-off — flagged True by the automated gate (NB1).`);
smart.push(`Treat ${{fmtPct(nb1.real_recall,2)}} real recall as the honest capacity ceiling for adversarial claims (NB2), not the in-sample 100% baseline.`);
if (nb2.drift_monitoring.any_alert) smart.push(`Feature/score drift alerts are active (NB2) — worst real PSI ${{psiWorstEntry[1].toFixed(4)}} on ${{psiWorstEntry[0]}}. Re-measure monthly in production.`);
smart.push(`Model classified Tier ${{nb6.model_tier}} (${{nb6.tier_description}}) — apply NB6's recommended oversight cadence.`);
smart.push(`Worst-case stress exposure is ${{rub.financial_exposure.stress_multiple.toFixed(1)}}&times; today's real cost (NB5 grid sweep, 105 combinations) — size contingency reserves accordingly.`);
if (!dq.all_checks_passed) smart.push(`BCBS 239 data-quality gate has failing checks (NB7) — resolve before relying on this pipeline for regulatory reporting.`);
smart.push(`${{allTbd}} of ${{allT}} governance sign-off checks (NB9) remain outside this pipeline's scope — route to real owners before go-live.`);
document.getElementById('smartList').innerHTML = smart.map(s => `<li>${{s}}</li>`).join("");
</script>
"""

##############################################################################
# WRITE OUTPUT
##############################################################################
_out_top = os.path.join(REPORTS_DIR, "Fraud_Detection_WorldClass_Dashboard.html")
_out_nb13 = os.path.join(RESULTS_DIR, "Fraud_Detection_WorldClass_Dashboard.html")
with open(_out_top, "w", encoding="utf-8") as f:
    f.write(html)
with open(_out_nb13, "w", encoding="utf-8") as f:
    f.write(html)

nb13_report = {
    "run_metadata": {"random_seed": RANDOM_SEED, "n_threads": _N_THREADS,
                      "generated_at_utc": datetime.now(timezone.utc).isoformat()},
    "sections_included": 17,
    "chart_types": ["bar", "doughnut", "radar", "polarArea", "line", "histogram-bar"],
    "html_bytes": len(html),
    "output_paths": [_out_top, _out_nb13],
}
with open(os.path.join(RESULTS_DIR, "nb13_report.json"), "w", encoding="utf-8") as f:
    json.dump(nb13_report, f, indent=2, default=str)

_ram_end = psutil.virtual_memory()
_total_elapsed = time.time() - _RUN_T0
print("=" * 70)
print(f"RAM at finish: {_ram_end.percent:.1f}% used ({_ram_end.used/1e9:.2f} GB / {_ram_end.total/1e9:.2f} GB)")
print(f"Total notebook wall-clock time: {_total_elapsed:.2f}s (real, measured).")
print(f"Notebook 13 complete. World-class dashboard written to: {_out_top}")
print(f"(also saved to: {_out_nb13})")
